# Module 5 - Session 4: Practical Exercises

**Objective:** To diagnose overfitting by plotting loss curves and to apply regularization techniques like Dropout.

**Structure used:** answer-only format with `Foundation -> Build -> Result`, plus runnable code for Exercise 2.


## Exercise 1: Interpreting Loss Curves (Conceptual)

### Foundation
- A **good fit** usually shows both training and validation loss decreasing and settling at similarly low values.
- **Overfitting** appears when training loss keeps improving but validation loss starts worsening.
- **Underfitting** appears when both losses stay high and fail to improve much.

### Build
1. **Model A diagnosis:** Good Fit
- Reason: both training and validation losses steadily decrease and converge near `0.1`.
- Concrete action: keep this setup, then use early stopping/checkpointing to save the best epoch.

2. **Model B diagnosis:** Overfitting
- Reason: training loss keeps dropping to `0.05`, but validation loss rises after around epoch 40.
- Concrete action: add regularization (e.g., Dropout) and/or stop training around epoch 40 with early stopping.

3. **Model C diagnosis:** Underfitting
- Reason: both losses get stuck high (`~0.8`) and do not improve.
- Concrete action: increase model capacity (more neurons/layers) or train longer with a better learning-rate setup.

### Result
- **Model A:** Good Fit -> preserve settings and checkpoint best model.
- **Model B:** Overfitting -> apply Dropout/early stopping.
- **Model C:** Underfitting -> increase capacity or improve optimization settings.


## Exercise 2: Applying Dropout (Coding)

This exercise uses TensorFlow/Keras on `make_moons` to compare a model without Dropout vs a model with Dropout.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

tf.random.set_seed(42)


In [ ]:
# Generate and prepare data
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Model WITHOUT Dropout
model_no_dropout = keras.models.Sequential([
    keras.layers.Input(shape=X_train.shape[1:]),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_no_dropout.compile(
    loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]
)

history1 = model_no_dropout.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    verbose=0,
)


In [ ]:
# Model WITH Dropout
model_with_dropout = keras.models.Sequential([
    keras.layers.Input(shape=X_train.shape[1:]),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

model_with_dropout.compile(
    loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]
)

history2 = model_with_dropout.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    verbose=0,
)


In [ ]:
# Compare training and validation loss curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history1.history["loss"], label="Train Loss")
plt.plot(history1.history["val_loss"], label="Val Loss")
plt.title("Without Dropout")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history2.history["loss"], label="Train Loss")
plt.plot(history2.history["val_loss"], label="Val Loss")
plt.title("With Dropout")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


### Observation (Markdown Answer)

### Foundation
- Dropout randomly disables a fraction of neurons during training, which reduces co-adaptation and improves generalization.

### Build
- **Without Dropout:** training loss often becomes very low, while validation loss may flatten or rise later, creating a wider train-vs-val gap.
- **With Dropout:** training loss is usually a bit higher, but validation loss is more stable and the gap between train and validation curves is typically smaller.

### Result
Dropout generally reduces overfitting by narrowing the training-validation loss gap and improving validation behavior over epochs.
